In [ ]:
import pandas as pd

In [ ]:
all_data = pd.read_parquet('../data/us_10m_nointernship_ai_skills_benefits.parquet.gzip')

In [ ]:
all_data_body = pd.read_parquet('../data/us_10m_nointernship_ai_skills_body.parquet.gzip')

# Exports

## Random Sample with Salary


In [ ]:
salary_data = all_data[all_data['SALARY'].notnull()]

In [ ]:
# salary_wham_data = salary_data[salary_data['wfh_wham'].notnull()]

In [ ]:
len(salary_data)

In [ ]:
salary_ai = salary_data[salary_data['AI ROLE'] == True]

In [ ]:
salary_no_ai = salary_data[salary_data['AI ROLE'] == False]

In [ ]:
salary_ai_sample = salary_ai.sample(n=10000, random_state=42)

In [ ]:
salary_no_ai_sample = salary_no_ai.sample(n=10000, random_state=42)

In [ ]:
salary_sample_all = pd.concat([salary_ai_sample, salary_no_ai_sample])

In [ ]:
# salary_sample_all = salary_sample_all.merge(body, left_on='ID', right_on='ID', how='left')

In [ ]:
len(salary_sample_all)

In [ ]:
salary_sample_all[salary_sample_all['wfh_wham_prob'].isna()]

In [ ]:
salary_sample_all.to_parquet('../data/salary_sample_2018_2023.parquet.gzip', compression='gzip')

## Random Sample with Salary and Remote Work


In [ ]:
salary_data = all_data[all_data['SALARY'].notnull()]

In [ ]:
salary_wham_data = salary_data[salary_data['wfh_wham'].notnull()]

In [ ]:
len(salary_wham_data)

In [ ]:
salary_ai = salary_wham_data[salary_wham_data['AI ROLE'] == True]

In [ ]:
salary_no_ai = salary_wham_data[salary_wham_data['AI ROLE'] == False]

In [ ]:
salary_ai_sample = salary_ai.sample(n=10000, random_state=42)

In [ ]:
salary_no_ai_sample = salary_no_ai.sample(n=10000, random_state=42)

In [ ]:
salary_sample_all = pd.concat([salary_ai_sample, salary_no_ai_sample])

In [ ]:
# salary_sample_all = salary_sample_all.merge(body, left_on='ID', right_on='ID', how='left')

In [ ]:
len(salary_sample_all)

In [ ]:
salary_sample_all[salary_sample_all['wfh_wham_prob'].isna()]

In [ ]:
salary_sample_all.to_parquet('../data/salary_wham_sample_20k.parquet.gzip', compression='gzip')

## Random Sample without Salary

In [ ]:
all_data_wham = all_data[all_data['wfh_wham'].notnull()]

In [ ]:
all_data_ai = all_data_wham[all_data_wham['AI ROLE'] == True]
all_data_no_ai = all_data_wham[all_data_wham['AI ROLE'] == False]

In [ ]:
all_data_ai_sample = all_data_ai.sample(n=10000, random_state=42)
all_data_no_ai_sample = all_data_no_ai.sample(n=10000, random_state=42)
all_data_sample_all = pd.concat([all_data_ai_sample, all_data_no_ai_sample])

In [ ]:
# all_data_sample_all = all_data_sample_all.merge(body, left_on='ID', right_on='ID', how='left')

In [ ]:
all_data_sample_all.to_parquet('../data/nosalary_sample_20k.parquet.gzip', compression='gzip')

# Check Composition

## Sample All

In [ ]:
all_data_sample = pd.read_parquet('../data/nosalary_sample_body.parquet.gzip')

In [ ]:
all_data_sample

In [ ]:
salary_data_sample = pd.read_parquet('../data/salary_sample_body.parquet.gzip')

In [ ]:
industry_crosswalk = pd.read_csv('../../thesis/Resources/Expanded_NAICS_Code_to_Industry_Name_Crosswalk.csv')

In [ ]:
industry_crosswalk = industry_crosswalk.rename(columns = {'Industry name': 'BLS_industry'})

In [ ]:
industry_crosswalk.dropna(inplace=True)

In [ ]:
industry_crosswalk['NAICS code'] = industry_crosswalk['NAICS code'].astype(int)

In [ ]:
industry_crosswalk = industry_crosswalk.rename(columns = {'NAICS code': 'BLS_NAICS'})

In [ ]:
all_data_sample = all_data_sample.merge(industry_crosswalk, left_on = 'NAICS_2022_2', right_on = 'BLS_NAICS', how = 'left')

In [ ]:
all_data_sample.loc[all_data_sample['BLS_industry'].isna(), 'BLS_industry'] = all_data_sample.loc[all_data_sample['BLS_industry'].isna(), 'NAICS_2022_2_NAME']

In [ ]:
all_data_sample.groupby('BLS_industry', dropna=False).size()

In [ ]:
all_data_sample.to_parquet('../data/nosalary_sample_body.parquet.gzip', compression='gzip')

In [ ]:
def get_grouped_industry(data):
    grouped_industry = data.groupby('BLS_industry', dropna=False).size().reset_index(name='count')

    # Calculate the total count
    total_count = grouped_industry['count'].sum()

    # Calculate the percentage of each group
    grouped_industry['percentage'] = (grouped_industry['count'] / total_count) * 100

    return grouped_industry


In [ ]:
grouped_industry = get_grouped_industry(all_data_sample)

In [ ]:
bls_stats = pd.read_csv('../../thesis/Resources/Employment_by_Industry.csv')

In [ ]:
industry_comp = grouped_industry.merge(bls_stats, left_on = 'BLS_industry', right_on = 'Industry name', how = 'outer')

In [ ]:
industry_comp = industry_comp[['BLS_industry', 'Industry name','percentage', 'Percent distribution 2022']]

In [ ]:
industry_comp.rename(columns = {'percentage': 'OJV Data', 'Percent distribution 2022': 'National Statistics'}, inplace=True)

In [ ]:
industry_comp

In [ ]:
import matplotlib.pyplot as plt

# Filter out rows with missing values in either column
filtered_df = industry_comp.dropna(subset=['OJV Data', 'National Statistics'])

# Create a scatter plot
plt.figure(figsize=(12, 8))
plt.scatter(filtered_df['OJV Data'], filtered_df['National Statistics'], color='blue')

# Add labels for each point
for i, row in filtered_df.iterrows():
    plt.text(row['OJV Data'], row['National Statistics'], row['BLS_industry'], fontsize=9, ha='right')

# Set the labels and title
plt.xlabel('OJV Data')
plt.ylabel('National Statistics')
max_val = max(filtered_df['OJV Data'].max(), filtered_df['National Statistics'].max()) * 1

plt.plot([-0.5, max_val], [-0.5, max_val], linestyle='dotted', color='red')

# plt.title('Percentage vs National Statistics by BLS Industry')
plt.axis('equal')
# plt.xlim(0, max(filtered_df['OJV Data'].max(), filtered_df['National Statistics'].max()) * 1.1)
# plt.ylim(0, max(filtered_df['OJV Data'].max(), filtered_df['National Statistics'].max()) * 1.1)

# Show the plot
# Show the plot
plt.grid(True)
# save plot
plt.savefig('../figures/industry_comp_sampleall.png')
plt.show()


## Salary Sample

In [ ]:
salary_data_sample = salary_data_sample.merge(industry_crosswalk, left_on = 'NAICS_2022_2', right_on = 'BLS_NAICS', how = 'left')

In [ ]:
salary_data_sample.loc[salary_data_sample['BLS_industry'].isna(), 'BLS_industry'] = salary_data_sample.loc[salary_data_sample['BLS_industry'].isna(), 'NAICS_2022_2_NAME']

In [ ]:
salary_data_sample.groupby('BLS_industry', dropna=False).size()

In [ ]:
salary_data_sample.to_parquet('../data/salary_sample_body.parquet.gzip', compression='gzip')

In [ ]:
grouped_industry_salary = get_grouped_industry(salary_data_sample)

In [ ]:
industry_comp_salary = grouped_industry_salary.merge(bls_stats, left_on = 'BLS_industry', right_on = 'Industry name', how = 'outer')

In [ ]:
industry_comp_salary = industry_comp_salary[['BLS_industry', 'Industry name','percentage', 'Percent distribution 2022']]

In [ ]:
industry_comp_salary.rename(columns = {'percentage': 'OJV Data', 'Percent distribution 2022': 'National Statistics'}, inplace=True)

In [ ]:
industry_comp_salary

In [ ]:
import matplotlib.pyplot as plt

# Filter out rows with missing values in either column
filtered_df_salary = industry_comp_salary.dropna(subset=['OJV Data', 'National Statistics'])

# Create a scatter plot
plt.figure(figsize=(12, 8))
plt.scatter(filtered_df_salary['OJV Data'], filtered_df_salary['National Statistics'], color='blue')

# Add labels for each point
for i, row in filtered_df_salary.iterrows():
    plt.text(row['OJV Data'], row['National Statistics'], row['BLS_industry'], fontsize=9, ha='right')

# Set the labels and title
plt.xlabel('OJV Data')
plt.ylabel('National Statistics')
max_val = max(filtered_df_salary['OJV Data'].max(), filtered_df_salary['National Statistics'].max()) * 1

plt.plot([-0.5, max_val], [-0.5, max_val], linestyle='dotted', color='red')

# plt.title('Percentage vs National Statistics by BLS Industry')
plt.axis('equal')
# plt.xlim(0, max(filtered_df_salary['OJV Data'].max(), filtered_df_salary['National Statistics'].max()) * 1.1)
# plt.ylim(0, max(filtered_df_salary['OJV Data'].max(), filtered_df_salary['National Statistics'].max()) * 1.1)

# Show the plot
# Show the plot
plt.grid(True)
# save plot
plt.savefig('../figures/industry_comp_samplesalary.png')
plt.show()


# Export 1m sample